<a href="https://colab.research.google.com/github/salawhaaat/dl-svg-lora/blob/main/Salauat_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q datasets trl transformers accelerate peft bitsandbytes pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 49.3 MB/s eta 0:00:00


In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-5hx9b6hi/unsloth_98913098cd354c0498ba681ba5b4caff
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-5hx9b6hi/unsloth_98913098cd354c0498ba681ba5b4caff
  Resolved https://github.com/unslothai/unsloth.git to commit 39fe23ded85c2caa3f8407d2b61920af44175baa
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 117.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.8/403.8 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 120.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 114.8 MB/s eta 0:00:00

In [ ]:
import os, re, time, random, warnings
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from google.colab import drive

warnings.filterwarnings('ignore')
drive.mount('/content/drive')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

Mounted at /content/drive
Torch: 2.10.0+cu128
CUDA: True


In [ ]:
CONFIG = {
    "model_name": "unsloth/Qwen3-4B-bnb-4bit",
    "max_seq_length": 2048,
    "lora_r": 64,
    "lora_alpha": 128,
    "learning_rate": 2e-4,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 16,
    "gradient_accumulation_steps": 2,
    "warmup_steps": 100,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "eval_steps": 200,
    "save_steps": 500,
    "output_dir": "/content/drive/MyDrive/svg_competition/svg_lora_qwen3_4b_r64",
}

In [ ]:
train_df = pd.read_csv('/content/drive/MyDrive/svg_competition/train.csv')
test_df = pd.read_csv('/content/drive/MyDrive/svg_competition/test.csv')

train_df = train_df.dropna(subset=['prompt', 'svg'])
train_df = train_df[train_df['svg'].str.startswith('<svg')]

print(f"Clean rows: {len(train_df)}")

train_split, eval_split = train_test_split(train_df, test_size=0.02, random_state=SEED)
train_ds = Dataset.from_pandas(train_split[['prompt','svg']].reset_index(drop=True))
eval_ds = Dataset.from_pandas(eval_split[['prompt','svg']].reset_index(drop=True))
print(f"Train: {len(train_ds)}, Eval: {len(eval_ds)}")

Clean rows: 50000
Train: 49000, Eval: 1000


In [ ]:
SYSTEM_PROMPT = (
    "You are an expert SVG code generator. "
    "Generate valid SVG code based on the description.\n"
    "STRICT RULES:\n"
    "- Output ONLY raw SVG code, no explanation\n"
    "- Use: <svg xmlns=\"http://www.w3.org/2000/svg\" "
    "viewBox=\"0 0 256 256\" width=\"256\" height=\"256\">\n"
    "- ALLOWED tags: svg, g, path, rect, circle, ellipse, "
    "line, polyline, polygon, defs, use, symbol, clipPath, "
    "mask, linearGradient, radialGradient, stop, text, tspan, style\n"
    "- FORBIDDEN: script, foreignObject, image, animate\n"
    "- Max 8000 characters, max 256 path elements\n"
    "- Close all tags properly"
)

def format_sft_text(example):
    return {"text": (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['prompt']}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{example['svg']}<|im_end|>"
    )}

train_text = train_ds.map(format_sft_text, remove_columns=train_ds.column_names)
eval_text = eval_ds.map(format_sft_text, remove_columns=eval_ds.column_names)
print(train_text[0]["text"][:300])

Map:   0%|          | 0/49000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

<|im_start|>system
You are an expert SVG code generator. Generate valid SVG code based on the description.
STRICT RULES:
- Output ONLY raw SVG code, no explanation
- Use: <svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 256 256" width="256" height="256">
- ALLOWED tags: svg, g, path, rect, circl


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
model.print_trainable_parameters()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.18: Fast Qwen3 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.65G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/qwen3-4b-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.18 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


trainable params: 132,120,576 || all params: 4,154,588,672 || trainable%: 3.1801


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_steps=CONFIG["warmup_steps"],
    weight_decay=CONFIG["weight_decay"],
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG["logging_steps"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    report_to="none",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_text,
    eval_dataset=eval_text,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=True,
    args=training_args,
)

train_result = trainer.train(resume_from_checkpoint="/content/drive/MyDrive/svg_competition/svg_lora_qwen3_4b_r64/checkpoint-3500")
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])
print("Done!")
print(train_result)

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/49000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/1000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 49,000 | Num Epochs = 3 | Total steps = 4,596
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 2 x 1) = 32
 "-____-"     Trainable parameters = 132,120,576 of 4,154,588,672 (3.18% trained)


Step,Training Loss,Validation Loss
3600,0.224199,0.245552
3800,0.226455,0.244931
4000,0.218070,0.244466
4200,0.213793,0.244390
4400,0.217180,0.244239


Done!
TrainOutput(global_step=4596, training_loss=0.051970114959022914, metrics={'train_runtime': 4006.0934, 'train_samples_per_second': 36.694, 'train_steps_per_second': 1.147, 'total_flos': 3.400998784598016e+18, 'train_loss': 0.051970114959022914, 'epoch': 3.0})


In [ ]:
import os, re, time, warnings
import xml.etree.ElementTree as ET
import pandas as pd
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from google.colab import drive

warnings.filterwarnings('ignore')
drive.mount('/content/drive')

OUTPUT_DIR = "/content/drive/MyDrive/svg_competition/svg_lora_qwen3_4b_r64"
SUBMISSION_PATH = "/content/drive/MyDrive/svg_competition/submission_qwen3_4b_r64.csv"

SYSTEM_PROMPT = (
    "You are an expert SVG code generator. "
    "Generate valid SVG code based on the description.\n"
    "STRICT RULES:\n"
    "- Output ONLY raw SVG code, no explanation\n"
    "- Use: <svg xmlns=\"http://www.w3.org/2000/svg\" "
    "viewBox=\"0 0 256 256\" width=\"256\" height=\"256\">\n"
    "- ALLOWED tags: svg, g, path, rect, circle, ellipse, "
    "line, polyline, polygon, defs, use, symbol, clipPath, "
    "mask, linearGradient, radialGradient, stop, text, tspan, style\n"
    "- FORBIDDEN: script, foreignObject, image, animate\n"
    "- Max 8000 characters, max 256 path elements\n"
    "- Close all tags properly"
)

test_df = pd.read_csv('/content/drive/MyDrive/svg_competition/test.csv')
print(f"Test rows: {len(test_df)}")

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B")
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-4B",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
model = model.merge_and_unload()
model.eval()
print("Model ready")

SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", re.IGNORECASE)

def extract_svg(text):
    matches = list(SVG_REGEX.finditer(text))
    return matches[-1].group(0).strip() if matches else ""

def is_valid_svg(svg_text):
    if not svg_text:
        return False
    try:
        root = ET.fromstring(svg_text)
        return root.tag.endswith("svg")
    except ET.ParseError:
        return False

def fallback_svg():
    return '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><rect width="256" height="256" fill="white"/><circle cx="128" cy="128" r="64" fill="black"/></svg>'

def format_test_prompt(prompt):
    return f"Generate svg code for an image that looks like: {prompt} Don't use markdown just give svg code"

def generate_svg(prompt, max_new_tokens=2048):
    formatted_prompt = format_test_prompt(prompt)
    chat_text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{formatted_prompt}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )
    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True)
    svg = extract_svg(decoded)
    if not is_valid_svg(svg):
        return fallback_svg()
    return svg

Mounted at /content/drive
Test rows: 1000


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model ready


In [ ]:
BATCH_SIZE = 8
prompts_list = test_df["prompt"].tolist()
ids_list = test_df["id"].tolist()
tokenizer.padding_side = "left"

rows = []
invalid_count = 0
t0 = time.time()

for batch_start in range(0, len(prompts_list), BATCH_SIZE):
    batch_prompts = prompts_list[batch_start:batch_start+BATCH_SIZE]
    batch_ids = ids_list[batch_start:batch_start+BATCH_SIZE]

    formatted = [
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{format_test_prompt(p)}<|im_end|>\n"
        f"<|im_start|>assistant\n"
        for p in batch_prompts
    ]

    inputs = tokenizer(
        formatted,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    ).to(model.device)

    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    for j, (out_ids, row_id) in enumerate(zip(output_ids, batch_ids)):
        new_tokens = out_ids[input_len:]
        decoded = tokenizer.decode(new_tokens, skip_special_tokens=True)
        svg = extract_svg(decoded)
        if not is_valid_svg(svg):
            invalid_count += 1
            svg = fallback_svg()
            status = "FALLBACK"
        else:
            status = "OK"
        rows.append({"id": row_id, "svg": svg})
        print(f"{batch_start+j}/1000 | {(time.time()-t0)/60:.1f} min | invalid: {invalid_count} | {status}")

sub_df = pd.DataFrame(rows)
sub_df.to_csv(SUBMISSION_PATH, index=False)
print(f"Done! {(time.time()-t0)/60:.1f} min | Invalid: {invalid_count}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


0/1000 | 1.4 min | invalid: 1 | FALLBACK
1/1000 | 1.4 min | invalid: 1 | OK
2/1000 | 1.4 min | invalid: 2 | FALLBACK
3/1000 | 1.4 min | invalid: 2 | OK
4/1000 | 1.4 min | invalid: 2 | OK
5/1000 | 1.4 min | invalid: 2 | OK
6/1000 | 1.4 min | invalid: 3 | FALLBACK
7/1000 | 1.4 min | invalid: 3 | OK
8/1000 | 2.8 min | invalid: 4 | FALLBACK
9/1000 | 2.8 min | invalid: 4 | OK
10/1000 | 2.8 min | invalid: 4 | OK
11/1000 | 2.8 min | invalid: 4 | OK
12/1000 | 2.8 min | invalid: 4 | OK
13/1000 | 2.8 min | invalid: 4 | OK
14/1000 | 2.8 min | invalid: 5 | FALLBACK
15/1000 | 2.8 min | invalid: 6 | FALLBACK
16/1000 | 4.1 min | invalid: 6 | OK
17/1000 | 4.1 min | invalid: 6 | OK
18/1000 | 4.1 min | invalid: 6 | OK
19/1000 | 4.1 min | invalid: 7 | FALLBACK
20/1000 | 4.1 min | invalid: 7 | OK
21/1000 | 4.1 min | invalid: 8 | FALLBACK
22/1000 | 4.1 min | invalid: 8 | OK
23/1000 | 4.1 min | invalid: 8 | OK
24/1000 | 5.5 min | invalid: 8 | OK
25/1000 | 5.5 min | invalid: 9 | FALLBACK
26/1000 | 5.5 min | 